In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("final_weather.csv")
df

,state,district,year,month,precipitation_sum,relative_humidity_2m_mean,latitude,longitude,temperature_2m_mean,wind_speed_10m_mean
0,Andhra Pradesh,ANANTAPUR,2000,1,0.000000,52.903226,14.724220,77.430674,24.322581,10.541935
1,Andhra Pradesh,ANANTAPUR,2000,2,3.106897,51.931034,14.724220,77.430674,26.262069,9.410345
2,Andhra Pradesh,ANANTAPUR,2000,3,0.003226,41.387097,14.724220,77.430674,28.070968,8.593548
3,Andhra Pradesh,ANANTAPUR,2000,4,0.113333,41.700000,14.724220,77.430674,31.843333,10.970000
4,Andhra Pradesh,ANANTAPUR,2000,5,1.596774,56.000000,14.724220,77.430674,29.806452,14.177419
...,...,...,...,...,...,...,...,...,...,...
89239,West Bengal,PURULIA,2011,8,13.438710,89.064516,23.202097,86.322899,26.148387,8.016129
89240,West Bengal,PURULIA,2011,9,11.706667,87.966667,23.202097,86.322899,25.820000,8.640000
89241,West Bengal,PURULIA,2011,10,1.425806,77.516129,23.202097,86.322899,25.212903,4.812903
89242,West Bengal,PURULIA,2011,11,0.030000,69.533333,23.202097,86.322899,21.540000,5.610000


In [3]:
# Load weather data_set to convert it from month wise to season wise
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
# Assign season and season_year (separating Winter and Rabi)
def assign_season(row):
    y, m = row['year'], row['month']
    
    if m in [11, 12]:  
        return pd.Series({'season': 'Winter', 'season_year': y })
    elif m == 1:
        return pd.Series({'season': 'Winter', 'season_year': y-1})
    elif m in [2, 3]:
        return pd.Series({'season': 'Rabi', 'season_year': y})
    elif m in [4, 5]:
        return pd.Series({'season': 'Summer', 'season_year': y})
    elif m in [6, 7, 8]:
        return pd.Series({'season': 'Kharif', 'season_year': y})
    elif m in [9, 10]:
        return pd.Series({'season': 'Autumn', 'season_year': y})
    else:
        return pd.Series({'season': None, 'season_year': None})

df[['season', 'season_year']] = df.apply(assign_season, axis=1)
# Whole Year: feb of current year to jan of next year
df['whole_year'] = df.apply(
    lambda row: row['year'] if row['month'] in [2,3,4,5,6,7,8,9,10,11,12] else row['year'] - 1,
    axis=1
)

whole_df = df.groupby(['state', 'district', 'whole_year']).agg({
    'temperature_2m_mean': 'mean',
    'precipitation_sum':'mean',
    'relative_humidity_2m_mean':'mean',
    'wind_speed_10m_mean':'mean',
    'latitude': 'first',
    'longitude': 'first'
}).reset_index().rename(columns={'whole_year': 'season_year'})
whole_df['season'] = 'Whole Year'

# Seasonal aggregation
seasonal_df = df.groupby(['state', 'district', 'season_year', 'season']).agg({
    'temperature_2m_mean': 'mean',
    'precipitation_sum':'mean',
    'relative_humidity_2m_mean':'mean',
    'wind_speed_10m_mean':'mean',
    'latitude': 'first',
    'longitude': 'first'
}).reset_index()

# Merge all
final_df = pd.concat([seasonal_df, whole_df], ignore_index=True)
final_df = final_df.sort_values(by=['state', 'district', 'season_year', 'season']).reset_index(drop=True)

# Preview
final_df


,state,district,season_year,season,temperature_2m_mean,precipitation_sum,relative_humidity_2m_mean,wind_speed_10m_mean,latitude,longitude
0,Andhra Pradesh,ANANTAPUR,1999,Whole Year,24.322581,0.000000,52.903226,10.541935,14.724220,77.430674
1,Andhra Pradesh,ANANTAPUR,1999,Winter,24.322581,0.000000,52.903226,10.541935,14.724220,77.430674
2,Andhra Pradesh,ANANTAPUR,2000,Autumn,25.463495,3.328387,74.602151,9.585108,14.724220,77.430674
3,Andhra Pradesh,ANANTAPUR,2000,Kharif,25.849892,3.307706,72.830108,19.102079,14.724220,77.430674
4,Andhra Pradesh,ANANTAPUR,2000,Rabi,27.166518,1.555061,46.659066,9.001947,14.724220,77.430674
...,...,...,...,...,...,...,...,...,...,...
45877,West Bengal,PURULIA,2011,Kharif,27.346165,10.682832,83.571685,8.512258,23.202097,86.322899
45878,West Bengal,PURULIA,2011,Rabi,23.972638,0.429032,48.365783,7.387212,23.202097,86.322899
45879,West Bengal,PURULIA,2011,Summer,29.388280,2.743602,60.327419,7.153280,23.202097,86.322899
45880,West Bengal,PURULIA,2011,Whole Year,25.353461,4.686931,70.221834,7.258359,23.202097,86.322899


In [4]:
# load crop dataset to merge it  with weather dataset
agriculture=pd.read_csv('updated_crop.csv')

agriculture

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2001,Kharif,Arecanut,1254.0,2061.00
1,Andaman and Nicobar Islands,NICOBARS,2001,Kharif,Rice,83.0,300.00
2,Andaman and Nicobar Islands,NICOBARS,2001,Whole Year,Cashewnut,719.0,192.00
3,Andaman and Nicobar Islands,NICOBARS,2001,Whole Year,Coconut,18190.0,64430000.00
4,Andaman and Nicobar Islands,NICOBARS,2002,Kharif,Rice,189.2,510.84
...,...,...,...,...,...,...,...
105570,West Bengal,PURULIA,2010,Rabi,Wheat,2013.0,5152.00
105571,West Bengal,PURULIA,2010,Summer,Maize,258.0,391.00
105572,West Bengal,PURULIA,2010,Summer,Rice,105.0,281.00
105573,West Bengal,PURULIA,2010,Whole Year,Sugarcane,303.0,11541.00


In [5]:
# checking for null value in crop dataset
agriculture.isnull().sum()

State_Name          0
District_Name       0
Crop_Year           0
Season              0
Crop                0
Area                0
Production       2440
dtype: int64

In [6]:

agriculture.columns

Index(['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area',
       'Production'],
      dtype='str')

In [7]:
final_df.columns

Index(['state', 'district', 'season_year', 'season', 'temperature_2m_mean',
       'precipitation_sum', 'relative_humidity_2m_mean', 'wind_speed_10m_mean',
       'latitude', 'longitude'],
      dtype='str')

In [8]:
# merging both dataset
# Ensure consistent column naming and case
agriculture['State_Name'] =agriculture['State_Name'].str.strip().str.lower()
agriculture['District_Name'] = agriculture['District_Name'].str.strip().str.lower()
agriculture['Season'] = agriculture['Season'].str.strip().str.lower()

final_df['state'] = final_df['state'].str.strip().str.lower()
final_df['district'] = final_df['district'].str.strip().str.lower()
final_df['season'] = final_df['season'].str.strip().str.lower()

# Merge on the matching keys
merged_df = pd.merge(
    agriculture,
    final_df,
    left_on=['State_Name', 'District_Name', 'Crop_Year', 'Season'],
    right_on=['state', 'district', 'season_year', 'season'],
    how='left'
)

# Drop extra matching columns (state, district, etc.)
merged_df = merged_df.drop(columns=['state', 'district', 'season_year', 'season'])

# Optional: reorder or check columns
print(merged_df.columns)
merged_df.head()

Index(['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area',
       'Production', 'temperature_2m_mean', 'precipitation_sum',
       'relative_humidity_2m_mean', 'wind_speed_10m_mean', 'latitude',
       'longitude'],
      dtype='str')


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,temperature_2m_mean,precipitation_sum,relative_humidity_2m_mean,wind_speed_10m_mean,latitude,longitude
0,andaman and nicobar islands,nicobars,2001,kharif,Arecanut,1254.0,2061.00,NaN,NaN,NaN,NaN,NaN,NaN
1,andaman and nicobar islands,nicobars,2001,kharif,Rice,83.0,300.00,NaN,NaN,NaN,NaN,NaN,NaN
2,andaman and nicobar islands,nicobars,2001,whole year,Cashewnut,719.0,192.00,NaN,NaN,NaN,NaN,NaN,NaN
3,andaman and nicobar islands,nicobars,2001,whole year,Coconut,18190.0,64430000.00,NaN,NaN,NaN,NaN,NaN,NaN
4,andaman and nicobar islands,nicobars,2002,kharif,Rice,189.2,510.84,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:

# drop null value
merged_df=merged_df.dropna()
merged_df

In [10]:
len(merged_df['District_Name'].unique())

616

In [11]:

# final data saving of 2001-2010
merged_df.to_csv('data_set_of_2001-10.csv',index=False)